# Chemprop D-MPNN — Bicerano Polymers (Tg)

Confronto aggiuntivo con la baseline del progetto.

**Protocollo**
- dataset: `PolymerPrediction.biceranoPolymers`;
- target: `Tg`;
- ordinamento deterministico dei documenti MongoDB per `_id`;
- test set: 20% con `random_state=28`, come nella baseline;
- dal restante 80% viene ricavata una validation del 12.5% del development set, cioè circa il 10% del dataset totale, usata solo da Chemprop per early stopping/checkpoint;
- metriche finali sul test: **MAE, R², RMSE e MAE/σ**;
- nessuno scaffold split e nessun hyperparameter tuning aggiuntivo.

Il test set resta quindi direttamente confrontabile con quello della baseline.


In [ ]:
from pathlib import Path
import os
import sys
import shutil
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from pymongo import MongoClient
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from rdkit import Chem

RANDOM_STATE = 28
DB_NAME = "PolymerPrediction"
COLLECTION_NAME = "polyInfo_kaggle"
TARGET = "Tg"

PROJECT_NAME = "polymer_prediction"
ROOT = Path.cwd()

while ROOT != ROOT.parent and ROOT.name != PROJECT_NAME:
    ROOT = ROOT.parent

if ROOT.name != PROJECT_NAME:
    raise RuntimeError(f"Root del progetto {PROJECT_NAME!r} non trovata")

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "results" / "chemprop" / f"chemprop_{COLLECTION_NAME}" 

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Caricamento del dataset

I documenti vengono ordinati per `_id` **prima** di costruire il DataFrame.  
Questo evita che l'ordine non garantito di `collection.find()` modifichi lo split anche mantenendo fisso `random_state`.


In [ ]:
load_dotenv(ROOT / ".env")

MONGO_URI = os.getenv("MONGO_URI")
if not MONGO_URI:
    raise RuntimeError("MONGO_URI non trovato. Controlla il file .env.")

client = MongoClient(MONGO_URI)
collection = client[DB_NAME][COLLECTION_NAME]

# Ordinamento deterministico: deve essere lo stesso usato dalla baseline.
docs = list(collection.find({}).sort("_id", 1))
df = pd.DataFrame(docs)

if df.empty:
    raise RuntimeError(f"La collection {DB_NAME}.{COLLECTION_NAME} è vuota.")

# Manteniamo l'id come riferimento per verificare l'allineamento dei campioni.
df["sample_id"] = df["_id"].astype(str)

# Espande properties quando il campo e presente.
if "properties" in df.columns:
    properties = pd.json_normalize(df["properties"])
    df = pd.concat(
        [df.drop(columns=["properties"]), properties],
        axis=1,
    )

required = {"smiles", TARGET, "sample_id"}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Colonne mancanti nel dataset: {sorted(missing)}")

dataset = df[["sample_id", "smiles", TARGET]].copy()

dataset[TARGET] = pd.to_numeric(dataset[TARGET], errors="coerce")
dataset = dataset.dropna(subset=["smiles", TARGET]).reset_index(drop=True)

# Le stringhe vuote non sono input validi.
dataset["smiles"] = dataset["smiles"].astype(str).str.strip()
dataset = dataset[dataset["smiles"] != ""].reset_index(drop=True)

# Scarta gli SMILES non validi
n_before = len(dataset)
dataset = dataset[~dataset["smiles"].str.contains(r"\[\[", na=False)].copy()

print("Righe eliminate per '[[':", n_before - len(dataset))

# Controlla che RDKit riesca a interpretare lo SMILES.
valid_mask = dataset["smiles"].apply(lambda s: Chem.MolFromSmiles(s) is not None)
invalid_smiles = dataset.loc[~valid_mask].copy()
print("Righe eliminate perché non valide per RDKit:", len(invalid_smiles))

if not invalid_smiles.empty:
    display(invalid_smiles)
    invalid_smiles.to_csv(OUTPUT_DIR / "invalid_rdkit_smiles.csv", index=False)

dataset = dataset.loc[valid_mask].reset_index(drop=True)

assert dataset["sample_id"].is_unique, "sample_id duplicati: controllare il dataset."

print(f"Campioni utilizzabili: {len(dataset)}")
print(f"{TARGET} media: {dataset[TARGET].mean():.2f} K")
print(f"{TARGET} sigma: {dataset[TARGET].std(ddof=1):.2f} K")
display(dataset.head())


## 2. Split train / validation / test

Prima viene creato lo stesso hold-out 80/20 della baseline.  
Solo dopo, il development set dell'80% viene diviso in train/validation per Chemprop.

La validation non entra mai nella valutazione finale.


In [ ]:
indices = np.arange(len(dataset))

dev_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

# 12.5% dell'80% = 10% del dataset complessivo.
train_idx, val_idx = train_test_split(
    dev_idx,
    test_size=0.125,
    random_state=RANDOM_STATE,
)

dataset["split"] = ""

dataset.loc[train_idx, "split"] = "train"
dataset.loc[val_idx, "split"] = "val"
dataset.loc[test_idx, "split"] = "test"

assert (dataset["split"] != "").all()
assert set(dataset["split"]) == {"train", "val", "test"}

print(dataset["split"].value_counts())
print()
print(dataset["split"].value_counts(normalize=True).sort_index())

# Salvo anche gli ID dei tre split per poterli confrontare con altri esperimenti.
for split_name in ("train", "val", "test"):
    ids = dataset.loc[dataset["split"] == split_name, "sample_id"]
    ids.to_csv(
        OUTPUT_DIR / f"{split_name}_sample_ids.csv",
        index=False,
        header=["sample_id"],
    )

chemprop_data_path = DATA_DIR / "biceranoPolymers_chemprop.csv"
dataset.to_csv(chemprop_data_path, index=False)

print(f"\nDataset Chemprop: {chemprop_data_path.resolve()}")


## 3. Training Chemprop

Configurazione:
- D-MPNN standard;
- hidden size 300;
- depth 3;
- dropout 0.3;
- batch size 16;
- massimo 120 epoche;
- early stopping sulla validation;
- un solo run.

Le metriche richieste a Chemprop sono MAE, R² e RMSE; il **MAE** è la prima metrica e viene quindi usato per il checkpointing.


In [ ]:
import importlib.metadata

chemprop_exe = shutil.which("chemprop")

if chemprop_exe is None:
    if os.name == "nt":
        candidate = Path(sys.prefix) / "Scripts" / "chemprop.exe"
    else:
        candidate = Path(sys.prefix) / "bin" / "chemprop"

    if not candidate.exists():
        raise RuntimeError(
            "Chemprop non trovato nell'ambiente corrente. "
            "Attiva l'ambiente in cui è installato Chemprop e riesegui il notebook."
        )

    chemprop_exe = str(candidate)

print("Chemprop executable:", chemprop_exe)
print("Chemprop version:", importlib.metadata.version("chemprop"))

subprocess.run([chemprop_exe, "--help"], check=True)

In [ ]:
cmd = [
    chemprop_exe,
    "train",
    "--data-path", str(chemprop_data_path),
    "--output-dir", str(OUTPUT_DIR),
    "--smiles-columns", "smiles",
    "--target-columns", TARGET,
    "--task-type", "regression",

    # Split definito esplicitamente sopra.
    "--splits-column", "split",

    # Metriche calcolate.
    "--metrics", "mae", "r2", "rmse",

    # MAE usato per early stopping e checkpoint.
    "--tracking-metric", "mae",

    # Architettura.
    "--message-hidden-dim", "300",
    "--depth", "3",
    "--ffn-hidden-dim", "300",
    "--ffn-num-layers", "1",
    "--dropout", "0.3",

    # Training.
    "--batch-size", "16",
    "--epochs", "120",
    "--patience", "20",

    # Riproducibilità.
    "--pytorch-seed", str(RANDOM_STATE),
]

print("Comando Chemprop:")
print(" ".join(map(str, cmd)))

stdout_path = OUTPUT_DIR / "train_stdout.log"
stderr_path = OUTPUT_DIR / "train_stderr.log"

with open(stdout_path, "w", encoding="utf-8") as f_out, \
     open(stderr_path, "w", encoding="utf-8") as f_err:

    result = subprocess.run(
        cmd,
        stdout=f_out,
        stderr=f_err,
        text=True,
    )

print("\nReturn code:", result.returncode)
print("STDOUT:", stdout_path.resolve())
print("STDERR:", stderr_path.resolve())

if result.returncode != 0:
    print("\n--- STDERR ---")
    print(stderr_path.read_text(encoding="utf-8"))
    raise RuntimeError(
        "Il training Chemprop è terminato con errore."
    )

## 4. Predizione sul test set

La predizione viene eseguita **solo** sui campioni assegnati al test esterno del 20%.


In [ ]:
test_df = (
    dataset.loc[dataset["split"] == "test", ["sample_id", "smiles", TARGET]]
    .copy()
    .reset_index(drop=True)
)

# Chemprop ha bisogno solo degli SMILES in inferenza.
test_input_path = DATA_DIR / "biceranoPolymers_chemprop_test.csv"
test_df[["smiles"]].to_csv(test_input_path, index=False)

pred_path = OUTPUT_DIR / "test_predictions.csv"

predict_cmd = [
    chemprop_exe,
    "predict",
    "--test-path", str(test_input_path),
    "--model-path", str(OUTPUT_DIR),
    "--smiles-columns", "smiles",
    "--preds-path", str(pred_path),
]

print("Comando prediction:")
print(" ".join(map(str, predict_cmd)))

result = subprocess.run(predict_cmd)

if result.returncode != 0:
    raise RuntimeError(
        f"La predizione Chemprop è terminata con errore: {result.returncode}"
    )

pred_df = pd.read_csv(pred_path)

if TARGET not in pred_df.columns:
    raise KeyError(
        f"Colonna di predizione '{TARGET}' non trovata. "
        f"Colonne disponibili: {pred_df.columns.tolist()}"
    )

assert len(pred_df) == len(test_df), "Numero di predizioni diverso dal numero di campioni test."

display(pred_df.head())


## 5. Metriche finali

`MAE/σ` usa la deviazione standard della Tg sull'intero dataset `biceranoPolymers`, come nel resto del lavoro.


In [ ]:
y_true = test_df[TARGET].to_numpy(dtype=float)
y_pred = pred_df[TARGET].to_numpy(dtype=float)

mae = mean_absolute_error(y_true, y_pred)
rmse = mean_squared_error(y_true, y_pred) ** 0.5
r2 = r2_score(y_true, y_pred)

sigma = dataset[TARGET].std(ddof=1)
mae_sigma = mae / sigma

summary = pd.DataFrame(
    [{
        "model": "Chemprop D-MPNN",
        "dataset": COLLECTION_NAME,
        "target": TARGET,
        "random_state": RANDOM_STATE,
        "n_total": len(dataset),
        "n_train": int((dataset["split"] == "train").sum()),
        "n_val": int((dataset["split"] == "val").sum()),
        "n_test": int((dataset["split"] == "test").sum()),
        "MAE_K": mae,
        "RMSE_K": rmse,
        "R2": r2,
        "sigma_K": sigma,
        "MAE_sigma": mae_sigma,
    }]
)

summary_path = OUTPUT_DIR / "summary.csv"
summary.to_csv(summary_path, index=False)

predictions_with_targets = test_df.copy()
predictions_with_targets["Tg_pred"] = y_pred
predictions_with_targets["abs_error"] = np.abs(y_true - y_pred)
predictions_with_targets.to_csv(
    OUTPUT_DIR / "test_predictions_with_targets.csv",
    index=False,
)

display(summary.round(4))
print(f"\nSummary salvato in: {summary_path.resolve()}")


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred, alpha=0.7)

low = min(y_true.min(), y_pred.min())
high = max(y_true.max(), y_pred.max())

plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Tg sperimentale [K]")
plt.ylabel("Tg predetta [K]")
plt.title(
    f"Chemprop D-MPNN — Bicerano\n"
    f"MAE = {mae:.1f} K | R² = {r2:.3f} | MAE/σ = {mae_sigma:.3f}"
)

plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "predicted_vs_true.png",
    dpi=200,
    bbox_inches="tight",
)
plt.show()
